# Proyecto 6. Análisis de resultados

FTL de comparación por plano anatómico: FedAvg vs FedProx

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RESULTS = Path('../results')
sns.set_theme(style='whitegrid', palette='tab10')
HOSPITALS = ['H1-axial', 'H2-coronal', 'H3-sagital', 'H4-mixto']

In [ ]:
def load(name):
    with open(RESULTS / f'{name}.json') as f:
        return json.load(f)

local      = load('local_only')
central    = load('centralized')
fedavg     = load('fedavg')
fedprox    = load('fedprox')

## 1. Tabla resumen de baselines vs federado

In [ ]:
rows = []

# Local only: AUC por hospital
for c in local['clients']:
    rows.append({'Método': 'Local-only', 'Hospital': HOSPITALS[c['hospital']-1],
                 'AUC': c['auc'], 'F1': c['f1'], 'Sensibilidad': c['sensitivity']})

# Centralizado global
cg = central['final_global']
rows.append({'Método': 'Centralizado', 'Hospital': 'Global',
             'AUC': cg['auc'], 'F1': cg['f1'], 'Sensibilidad': cg['sensitivity']})

# FedAvg y FedProx (promedio seeds)
for nombre, exp in [('FedAvg', fedavg), ('FedProx', fedprox)]:
    agg = exp['aggregated']
    rows.append({'Método': nombre, 'Hospital': 'Global',
                 'AUC': f"{agg['global_auc']['mean']:.4f} ± {agg['global_auc']['std']:.4f}",
                 'F1': f"{agg['global_f1']['mean']:.4f}",
                 'Sensibilidad': f"{agg['global_sen']['mean']:.4f}"})

df = pd.DataFrame(rows)
df

## 2. Curvas de entrenamiento. AUC global por ronda

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for nombre, exp, ax_idx in [('FedAvg', fedavg, 0), ('FedProx', fedprox, 1)]:
    ax = axes[ax_idx]
    for i, seed_res in enumerate(exp['seeds']):
        h = seed_res['history']
        ax.plot(h['round'], h['global_auc'], alpha=0.5, label=f'seed {i+1}')
    # Media entre seeds
    mean_auc = np.mean([s['history']['global_auc'] for s in exp['seeds']], axis=0)
    ax.plot(exp['seeds'][0]['history']['round'], mean_auc, 'k--', lw=2, label='media')
    ax.set_title(f'{nombre} — AUC global por ronda')
    ax.set_xlabel('Ronda')
    ax.set_ylabel('AUC-ROC')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS / 'curvas_auc.png', dpi=150)
plt.show()

## 3. Equidad entre hospitales (std AUC)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for nombre, exp, ls in [('FedAvg', fedavg, '-'), ('FedProx', fedprox, '--')]:
    mean_eq = np.mean([s['history']['equity_auc'] for s in exp['seeds']], axis=0)
    rounds   = exp['seeds'][0]['history']['round']
    ax.plot(rounds, mean_eq, ls=ls, label=nombre, lw=2)

ax.set_title('Equidad entre hospitales.std(AUC) por ronda')
ax.set_xlabel('Ronda')
ax.set_ylabel('std(AUC) entre clientes')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'equity.png', dpi=150)
plt.show()

## 4. AUC por hospital en la última ronda

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(4)
width = 0.35

for i, (nombre, exp) in enumerate([('FedAvg', fedavg), ('FedProx', fedprox)]):
    client_aucs = np.mean(
        [[s['final_clients'][k]['auc'] for k in range(4)] for s in exp['seeds']], axis=0
    )
    ax.bar(x + i*width, client_aucs, width, label=nombre)

ax.set_xticks(x + width/2)
ax.set_xticklabels(HOSPITALS)
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC por hospital (última ronda, media semillas)')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / 'auc_por_hospital.png', dpi=150)
plt.show()

## 5. Coste de comunicación

In [ ]:
n_params = fedavg['seeds'][0]['n_params']
comm_mb  = fedavg['seeds'][0]['comm_bytes_per_round'] / 1e6
total_mb = comm_mb * 20  # 20 rondas

print(f'Parámetros del modelo:       {n_params:,}')
print(f'Coste por ronda (4 clientes): {comm_mb*4:.2f} MB')
print(f'Coste total (20 rondas):     {total_mb*4:.2f} MB')